[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C11/E11-2_Variables_Redondantes.ipynb)

# Rugosité d'un procédé de tournage : écarter les variables redondantes

## Le contexte

Un atelier de tournage de finition relève, pour chaque passe, le matériau usiné, la vitesse de coupe, la vitesse de broche, le diamètre usiné, l'avance par tour, la vitesse d'avance et le rayon de bec de la plaquette. La rugosité Ra de la surface est ensuite mesurée au rugosimètre. On voudrait prévoir la rugosité d'après les réglages, pour choisir ceux qui la tiennent sous la tolérance.

Plusieurs colonnes disent la même chose. La vitesse de broche se déduit de la vitesse de coupe et du diamètre, la vitesse d'avance du produit de l'avance par tour et de la vitesse de broche. Garder toutes les colonnes alourdit le modèle sans l'éclairer : la sélection récursive de variables (RFECV) écarte celles qui n'apportent rien.

La grandeur à prédire est un nombre en µm : c'est un problème de régression.

Défi : prédire `rugosite_Ra_um` et obtenir un R² supérieur à celui du modèle de départ sur la base de test. Le modèle de départ est une régression linéaire sur les sept colonnes, cases vides remplacées par la médiane et variables mises à l'échelle ; il atteint un R² de 0,801 : à vous de faire mieux. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Variables_Redondantes.xlsx`, dans le dossier `donnees` du dépôt : 450 passes, 8 colonnes. Une ligne est une passe de finition au tour.

| Colonne | Signification | Unité |
|---|---|---|
| `materiau` | acier, inox ou aluminium | catégorie |
| `vitesse_coupe_m_min` | vitesse de coupe | m/min |
| `vitesse_broche_tr_min` | vitesse de broche, liée à la vitesse de coupe et au diamètre | tr/min |
| `diametre_mm` | diamètre usiné (quelques cases vides) | mm |
| `avance_mm_tr` | avance par tour | mm/tr |
| `avance_mm_min` | vitesse d'avance, produit de l'avance par tour et de la vitesse de broche | mm/min |
| `rayon_bec_mm` | rayon de bec de la plaquette (quelques cases vides) | mm |
| `rugosite_Ra_um` | cible : rugosité Ra mesurée | µm |

Origine : jeu simulé, rugosité théorique de tournage en f² / (32 r), avec une arête rapportée à basse vitesse de coupe et un coefficient propre au matériau. Il enseigne une méthode, et aucune conclusion de procédé n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et la répartition des matériaux : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Variables_Redondantes.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Variables_Redondantes.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "rugosite_Ra_um"
print("dimensions :", df.shape)
print("cases vides par colonne :")
print(df.isna().sum().to_string())
print(df["materiau"].value_counts().to_string())

Lecture du résultat : 450 passes, 8 colonnes. Deux colonnes ont des cases vides : le diamètre (21) et le rayon de bec (11). Les trois matériaux sont représentés, aucun n'est rare. Le matériau est une catégorie : il faudra la coder en nombres.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Variables_Redondantes.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : mettre le test de côté

Un quart des passes est mis de côté avant tout apprentissage, toute sélection et toute comparaison avec la cible. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=cible)
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=GRAINE)
print("apprentissage :", len(X_train), "passes")
print("test :", len(X_test), "passes")

### Étape 3 : vérifier que des colonnes se déduisent des autres

La vitesse de coupe est liée à la vitesse de broche et au diamètre par V = π D n / 1000. On calcule, sur les passes d'apprentissage dont le diamètre est connu, le rapport entre la vitesse de broche relevée et celle que donnerait la formule, puis le rapport entre la vitesse d'avance et le produit de l'avance par tour et de la vitesse de broche. Un rapport proche de 1 signale une colonne redondante.

In [ ]:
connu = X_train.dropna(subset=["diametre_mm"])
rapport_broche = 1000 * connu["vitesse_coupe_m_min"] / (np.pi * connu["diametre_mm"]) / connu["vitesse_broche_tr_min"]
rapport_avance = X_train["avance_mm_tr"] * X_train["vitesse_broche_tr_min"] / X_train["avance_mm_min"]
print(f"vitesse de broche, formule sur relevé : moyenne {rapport_broche.mean():.3f}, écart-type {rapport_broche.std():.3f}")
print(f"vitesse d'avance, produit sur relevé : moyenne {rapport_avance.mean():.3f}, écart-type {rapport_avance.std():.3f}")
correlations = X_train.corr(numeric_only=True)
print("corrélation vitesse de coupe et vitesse de broche :", round(correlations.loc["vitesse_coupe_m_min", "vitesse_broche_tr_min"], 2))
print("corrélation avance par tour et rugosité :", round(X_train["avance_mm_tr"].corr(y_train), 2))

Lecture du résultat : le rapport entre la vitesse de broche relevée et celle de la formule vaut en moyenne 1,000, avec un écart-type de 0,010 : la colonne n'apporte pas d'information propre. Il en va de même de la vitesse d'avance (rapport moyen 1,000, écart-type 0,003). La vitesse de coupe et la vitesse de broche sont corrélées à 0,94. Ces colonnes ne gênent pas un arbre, mais elles brouillent les coefficients d'un modèle linéaire.

### Étape 4 : construire le modèle de départ

Le modèle de départ est une régression linéaire sur les sept colonnes. La préparation code le matériau en colonnes de 0 et de 1 (la première modalité sert de référence), remplace les cases vides par la médiane de l'apprentissage et met les colonnes numériques à l'échelle. Le pipeline apprend sur `X_train` seul.

In [ ]:
# Votre code :

### Étape 5 : mesurer sur la base de test

Le modèle prédit des passes qu'il n'a jamais vues. On lit le coefficient de détermination R² et deux erreurs en µm, la racine de l'erreur quadratique moyenne (RMSE) et l'erreur absolue moyenne (MAE). Un témoin qui répond toujours la moyenne de l'apprentissage donne le point de comparaison le plus bas.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

y_pred = model.predict(X_test)
score_depart = r2_score(y_test, y_pred)
score_temoin = r2_score(y_test, np.full(len(y_test), y_train.mean()))
print(f"R² : {score_depart:.3f}")
print(f"RMSE : {root_mean_squared_error(y_test, y_pred):.2f} µm")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f} µm")
print(f"R² du témoin (toujours la moyenne de l'apprentissage) : {score_temoin:.3f}")
print(f"prédictions négatives : {int((y_pred < 0).sum())} sur {len(y_pred)}")

Lecture du résultat : le R² est de 0,801 et l'erreur absolue moyenne de 0,61 µm, pour une rugosité moyenne de 2,18 µm sur la base de test. Le témoin n'explique rien : son R² est nul, au signe de l'arrondi près. 11 prédictions sont négatives, ce qui n'a pas de sens physique pour une rugosité : la droite ne suit pas la forme de la loi.

## Pour aller plus loin : sélectionner, puis changer de forme

Le modèle de départ de l'étape 4 reste le score de référence du défi. Les deux cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première lance la sélection récursive RFECV avec un gradient boosting : elle écarte une colonne à la fois, la moins utile, et garde le nombre de colonnes qui maximise le R² en validation croisée sur l'apprentissage. La seconde compare par validation croisée répétée le modèle de départ, les colonnes retenues, le gradient boosting, un SVM de régression et une régression linéaire sur les logarithmes, qui correspond à une loi puissance. Le test ne sert qu'une fois, au modèle retenu. Les lectures sont imprimées par les cellules elles-mêmes.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

In [ ]:
# Votre code :

## Le score de référence

Le score publié est le R² du modèle de départ de l'étape 4 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = r2_score(y_test, model.predict(X_test))
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.3f" % score)

## Synthèse

1. Une colonne qui se déduit des autres, la vitesse de broche ou la vitesse d'avance, n'apporte aucune information et brouille les coefficients d'un modèle linéaire.
2. La sélection récursive écarte 2 colonnes sur les 8 colonnes codées (`diametre_mm`, `avance_mm_min`) sans perte de score en validation croisée.
3. Écarter des colonnes allège le modèle mais ne corrige pas sa forme : la régression linéaire garde le même niveau (0,765 avec sept colonnes, 0,765 avec les colonnes retenues, en validation croisée).
4. La loi est une puissance des réglages : un modèle linéaire sur les logarithmes atteint 0,967 en validation croisée, devant le gradient boosting (0,946) et le SVM de régression (0,955).
5. Le modèle retenu par la validation croisée est « régression linéaire sur les logarithmes (loi puissance) » ; mesuré une seule fois sur le test, il donne un R² de 0,984.